# Feature Effect Performance Regimes
This notebook assembles a run-scoped regime analysis table for one interpretable-model run, optionally evaluates reduced feature-effect representations, compares clustering behavior across performance groups, and exports lean per-candidate cluster artifacts for downstream inspection. The shared 2D UMAP visualization embedding is always built for cluster plots; optional UMAP reductions for clustering and trustworthiness can be disabled. Cluster inspection happens in `feature_effect_pr_cluster_inspection.ipynb`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from data_modelling.prepared_data import (
    MODEL_SETTING_COLS,
    load_prepared_data,
    prepare_single_target_model_data,
)
from data_modelling.run_context import (
    format_exported_model_label,
    get_exported_model_info,
    load_run_context,
)
from data_modelling.feature_effect_cluster_exports import write_cluster_exports
from data_modelling.feature_effect_performance_regimes_utils import (
    TRUSTWORTHINESS_COLUMNS,
    assemble_step1_analysis_table,
    assert_columns_present,
    build_feature_effect_regime_artifact_names,
    build_feature_effect_regime_export_layout,
    evaluate_umap_trustworthiness_by_group,
    get_effect_cols,
    load_or_initialize_feature_effect_regime_manifest,
    merge_feature_effect_regime_artifact_records,
    resolve_cluster_spec,
    resolve_feature_effect_regime_export_context,
    resolve_raw_metric_col,
    run_step2_clustering,
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

MODEL_ID = "gam"
RUN_NAME = "full_trainval_12ep_1seed"
EVAL_CSV_NAME = "eval_epoch_12.csv"
TARGET_COL = None  # Optional override, e.g. 'ml_ade_log'
LOWER_IS_BETTER = True
PERFORMANCE_GROUP_COL = "performance_group"

CLUSTER_SPEC = {
    "groups": ["easy", "medium", "hard"],
    "algorithms": ["hdbscan", "optics"],
    # Set False to skip optional UMAP reductions for clustering and trustworthiness.
    # The required 2D visualization UMAP is still computed for cluster plots.
    "evaluate_umap_latent_space": False,
    # Used only when evaluate_umap_latent_space=True. Set one integer for all groups
    # or a dict like {'easy': 2, 'medium': 4, 'hard': 2}.
    "umap_selected_n_components": {"easy": 3, "medium": 3, "hard": 3},
    "trustworthiness_neighbor_values": [5, 10, 15],
    "cluster_umap_n_neighbors": 30,
    "cluster_umap_min_dist": 0.0,
    "viz_umap_n_neighbors": 15,
    "viz_umap_min_dist": 0.1,
    "random_state": 42,
    # Set one integer for all groups or a dict like {'easy': 6, 'medium': 12, 'hard': 6}.
    "min_cluster_size": 5,
    # Set one integer for all groups or a dict like {'easy': 6, 'medium': 12, 'hard': 6}.
    "min_samples": 5,
    "optics_cluster_method": "xi",
    "optics_xi": 0.05,
    "distance_metric": "euclidean",
}


## Resolve Run Context and Artifact Paths
**Purpose:** Tie every input and output to one exported modelling run and one joined-metrics file.<br>
**Inputs:** `MODEL_ID`, `RUN_NAME`, `EVAL_CSV_NAME`, optional `TARGET_COL`.<br>
**Outputs:** Resolved run metadata, source artifact paths, and result directories for this notebook run.<br>
**How to Verify:** Confirm the printed target, feature count, model label, and source paths match the exported run you intend to analyze.


In [ ]:
run_ctx = load_run_context(MODEL_ID, RUN_NAME, TARGET_COL)
manifest = run_ctx.manifest
target_col = run_ctx.target_col
feature_cols = run_ctx.feature_cols
exported_model_info = get_exported_model_info(manifest)
exported_model_label = format_exported_model_label(exported_model_info)
target_mode = exported_model_info["target_mode"]
raw_metric_col = resolve_raw_metric_col(manifest, target_col)

PREPARED_DATA_PATH = (
    Path("../../results/interpretable_model/prepared_data")
    / RUN_NAME
    / f"prepared_data_{raw_metric_col}.csv"
)
FEATURE_EFFECTS_PATH = run_ctx.tables_dir / f"feature_effects_{target_col}.csv"
FEATURE_EFFECT_IMPORTANCE_PATH = (
    run_ctx.tables_dir / f"feature_effect_importance_{target_col}.csv"
)
JOINED_METRICS_PATH = (
    Path("../../results/trajectory_prediction/trajectory_metrics_joined")
    / RUN_NAME
    / EVAL_CSV_NAME
)
FEATURE_EFFECT_REGIME_RESULTS_ROOT = Path(
    "../../results/interpretable_model/feature_effect_performance_regimes"
)

required_paths = [
    ("prepared data export", PREPARED_DATA_PATH),
    ("feature-effect export", FEATURE_EFFECTS_PATH),
    ("feature-effect importance export", FEATURE_EFFECT_IMPORTANCE_PATH),
    ("joined metrics export", JOINED_METRICS_PATH),
]
for label, path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Missing required {label}: {path}")

print(f"Run: {RUN_NAME}")
print(f"Eval CSV: {EVAL_CSV_NAME}")
print(f"Exported model: {exported_model_label}")
print(f"Model ID: {MODEL_ID} | target_mode: {target_mode}")
print(f"Resolved target_col: {target_col}")
print(f"Resolved raw_metric_col: {raw_metric_col}")
print(f"Feature count: {len(feature_cols)}")
print(f"Prepared data path: {PREPARED_DATA_PATH}")
print(f"Feature-effect path: {FEATURE_EFFECTS_PATH}")
print(f"Feature-effect ranking path: {FEATURE_EFFECT_IMPORTANCE_PATH}")
print(f"Joined metrics path: {JOINED_METRICS_PATH}")
print(
    f"Feature-effect regime export root: {FEATURE_EFFECT_REGIME_RESULTS_ROOT.resolve()}"
)

## Load the Prepared Modelling Table
**Purpose:** Reconstruct the exact modelling rows and feature key used by the interpretable model.<br>
**Inputs:** `PREPARED_DATA_PATH`, resolved `target_col`, resolved `raw_metric_col`, and manifest `feature_cols`.<br>
**Outputs:** `model_df` on the notebook's modelling rows plus a verified feature-key definition.<br>
**How to Verify:** Check that the prepared target and feature columns exactly match the run manifest and that the printed row count looks plausible.


In [ ]:
prepared_df = load_prepared_data(
    PREPARED_DATA_PATH,
    display_fn=display,
    include_missing_summary=True,
    include_dtype_summary=True,
)

prepared = prepare_single_target_model_data(
    prepared_df,
    target_col=target_col,
    default_target=raw_metric_col,
    include_model_settings_as_features=any(
        col in MODEL_SETTING_COLS for col in feature_cols
    ),
)
model_df = prepared["model_df"].copy()
prepared_feature_cols = prepared["feature_cols"]

if prepared["target_col"] != target_col:
    raise ValueError(
        f"Prepared target mismatch. expected={target_col}, actual={prepared['target_col']}"
    )
if prepared_feature_cols != feature_cols:
    raise ValueError(
        "Prepared feature columns do not match the run manifest exactly. "
        f"expected={feature_cols}, actual={prepared_feature_cols}"
    )

print(f"Prepared modelling rows: {len(model_df)}")
print(f"Prepared feature key size: {len(prepared_feature_cols)}")

## Assemble the Performance-Regime Table
**Purpose:** Join prepared rows, run-scoped metrics, and feature-effect exports, then validate the clustering configuration against the loaded feature-effect columns.<br>
**Inputs:** `model_df`, `JOINED_METRICS_PATH`, `FEATURE_EFFECTS_PATH`, `FEATURE_EFFECT_IMPORTANCE_PATH`, and `CLUSTER_SPEC`.<br>
**Outputs:** `analysis_df`, performance-group summaries, ordered global ranking metadata, resolved clustering config, and the export paths used by the notebook.<br>
**How to Verify:** Confirm the join row counts, the detected feature-effect count, whether optional UMAP reductions are enabled, and the export root before moving on.


In [ ]:
def build_trustworthiness_plot_titles(
    cluster_spec: dict[str, object],
) -> dict[str, str]:
    plot_titles = {
        f"nn_{neighbor_value}": (
            "UMAP trustworthiness by reduced dimension and performance group "
            f"(n_neighbors={neighbor_value})"
        )
        for neighbor_value in cluster_spec["trustworthiness_neighbor_values"]
    }
    neighbor_label = ", ".join(
        str(value) for value in cluster_spec["trustworthiness_neighbor_values"]
    )
    plot_titles[cluster_spec["trustworthiness_mean_view"]] = (
        "UMAP trustworthiness by reduced dimension and performance group "
        f"(mean over n_neighbors={neighbor_label})"
    )
    return plot_titles


joined_metrics_df = pd.read_csv(JOINED_METRICS_PATH)
feature_effects_df = pd.read_csv(FEATURE_EFFECTS_PATH)
feature_effect_ranking_df = pd.read_csv(FEATURE_EFFECT_IMPORTANCE_PATH)
assert_columns_present(
    feature_effect_ranking_df,
    [
        "feature",
        "global_rank",
        "importance_metric",
        "importance_value",
        "importance_ascending",
    ],
    df_name="feature-effect ranking export",
)
feature_effect_ranking_df = feature_effect_ranking_df.sort_values(
    ["global_rank", "feature"]
).reset_index(drop=True)

analysis_df, group_summary_df = assemble_step1_analysis_table(
    prepared_model_df=model_df,
    joined_metrics_df=joined_metrics_df,
    feature_effects_df=feature_effects_df,
    feature_cols=feature_cols,
    target_col=target_col,
    performance_metric_col=raw_metric_col,
    lower_is_better=LOWER_IS_BETTER,
    performance_group_col=PERFORMANCE_GROUP_COL,
    random_state=CLUSTER_SPEC["random_state"],
)

effect_cols = get_effect_cols(analysis_df)
CLUSTER_SPEC_RESOLVED = resolve_cluster_spec(CLUSTER_SPEC, effect_cols=effect_cols)
EXPORT_CONTEXT = resolve_feature_effect_regime_export_context(
    model_id=MODEL_ID,
    run_name=RUN_NAME,
    target_col=target_col,
    eval_csv_name=EVAL_CSV_NAME,
    lower_is_better=LOWER_IS_BETTER,
    performance_group_col=PERFORMANCE_GROUP_COL,
    results_root=FEATURE_EFFECT_REGIME_RESULTS_ROOT,
)
EXPORT_LAYOUT = build_feature_effect_regime_export_layout(
    export_context=EXPORT_CONTEXT,
    cluster_spec=CLUSTER_SPEC_RESOLVED,
)
ARTIFACT_NAMES = build_feature_effect_regime_artifact_names(
    cluster_spec=CLUSTER_SPEC_RESOLVED
)
TABLES_DIR = EXPORT_LAYOUT["tables_dir"]
PLOTS_DIR = EXPORT_LAYOUT["plots_dir"]
EXPORT_MANIFEST_PATH = EXPORT_LAYOUT["manifest_path"]

REGIME_ANALYSIS_PATH = TABLES_DIR / ARTIFACT_NAMES["tables"]["regime_analysis"]
PERFORMANCE_GROUP_SUMMARY_PATH = (
    TABLES_DIR / ARTIFACT_NAMES["tables"]["performance_group_summary"]
)
UMAP_TRUSTWORTHINESS_PATH = (
    TABLES_DIR / ARTIFACT_NAMES["tables"]["umap_trustworthiness"]
)
CLUSTER_SCORES_PATH = TABLES_DIR / ARTIFACT_NAMES["tables"]["cluster_scores"]
CLUSTER_ASSIGNMENTS_PATH = TABLES_DIR / ARTIFACT_NAMES["tables"]["cluster_assignments"]
CLUSTER_FEATURE_EFFECT_PROFILES_PATH = (
    TABLES_DIR / ARTIFACT_NAMES["tables"]["cluster_feature_effect_profiles"]
)
CLUSTER_CATALOG_PATH = TABLES_DIR / ARTIFACT_NAMES["tables"]["cluster_catalog"]
FEATURE_EFFECT_GLOBAL_RANKING_PATH = (
    TABLES_DIR / ARTIFACT_NAMES["tables"]["feature_effect_global_ranking"]
)
RAW_ALGORITHM_GRID_PATH = (
    PLOTS_DIR / ARTIFACT_NAMES["plots"]["raw_algorithm_comparison_grid"]
)
UMAP_ALGORITHM_GRID_PATH = (
    PLOTS_DIR / ARTIFACT_NAMES["plots"]["umap_algorithm_comparison_grid"]
)
RAW_ALGORITHM_GRID_NO_NOISE_PATH = (
    PLOTS_DIR / ARTIFACT_NAMES["plots"]["raw_algorithm_comparison_grid_no_noise"]
)

if CLUSTER_SPEC_RESOLVED["evaluate_umap_latent_space"]:
    UMAP_TRUSTWORTHINESS_PLOT_PATHS = {
        trustworthiness_view: PLOTS_DIR / plot_name
        for trustworthiness_view, plot_name in ARTIFACT_NAMES["plots"][
            "umap_trustworthiness_curves"
        ].items()
    }
    TRUSTWORTHINESS_PLOT_TITLES = build_trustworthiness_plot_titles(
        CLUSTER_SPEC_RESOLVED
    )
else:
    UMAP_TRUSTWORTHINESS_PLOT_PATHS = {}
    TRUSTWORTHINESS_PLOT_TITLES = {}

group_counts_df = (
    analysis_df[PERFORMANCE_GROUP_COL]
    .value_counts()
    .rename_axis(PERFORMANCE_GROUP_COL)
    .reset_index(name="count")
)

print(f"Joined metrics rows: {len(joined_metrics_df)}")
print(f"Feature-effect rows: {len(feature_effects_df)}")
print(f"Feature ranking rows: {len(feature_effect_ranking_df)}")
print(f"Analysis rows: {len(analysis_df)}")
print(f"Feature-effect columns available: {len(effect_cols)}")
print(
    "Optional UMAP reductions for clustering/trustworthiness: "
    f"{CLUSTER_SPEC_RESOLVED['evaluate_umap_latent_space']}"
)
print(
    f"Resolved optional UMAP candidate dimensions: {CLUSTER_SPEC_RESOLVED['umap_candidate_dims']}"
)
print("2D visualization UMAP: always computed for cluster plots")
print(f"Cluster-spec export root: {EXPORT_LAYOUT['cluster_spec_root']}")
print(f"Export manifest path (manifest.json): {EXPORT_MANIFEST_PATH}")
display(group_summary_df)
display(group_counts_df)
display(feature_effect_ranking_df.head())
display(analysis_df.head())


## Visualize Performance Group Distribution
**Purpose:** Show the `ml_ade` distribution coloured by performance group on a linear (clipped) and log1p scale to verify that k-means found meaningful natural breaks.<br>
**Inputs:** `analysis_df`, `group_summary_df`, `raw_metric_col`, and `PERFORMANCE_GROUP_COL`.<br>
**Outputs:** Saved plot at `PERFORMANCE_GROUP_DISTRIBUTION_PATH`.<br>
**How to Verify:** Confirm that the dashed boundary lines align with visible transitions in the histogram and that the table counts match `group_summary_df`.

In [ ]:
import matplotlib.gridspec as gridspec
from matplotlib.colors import to_rgba

PERFORMANCE_GROUP_DISTRIBUTION_PATH = (
    PLOTS_DIR / f"performance_group_distribution_{raw_metric_col}.png"
)

_GROUP_COLORS = {"easy": "#4daf8d", "medium": "#f5c842", "hard": "#e8725a"}
_GROUP_ORDER = ["easy", "medium", "hard"]


def plot_performance_group_distribution(
    analysis_df: pd.DataFrame,
    *,
    performance_metric_col: str,
    performance_group_col: str,
    group_summary_df: pd.DataFrame,
    plot_path: Path,
) -> None:
    metric = analysis_df[performance_metric_col]
    n_total = len(metric)
    clip_val = float(metric.quantile(0.995))
    full_max = float(metric.max())
    boundary_low = float(group_summary_df["boundary_easy_medium"].iloc[0])
    boundary_high = float(group_summary_df["boundary_medium_hard"].iloc[0])

    range_strs = {
        "easy": f"<= {boundary_low:.2f}",
        "medium": f"({boundary_low:.2f}, {boundary_high:.2f}]",
        "hard": f"> {boundary_high:.2f}",
    }
    table_rows = []
    for g in _GROUP_ORDER:
        mask = analysis_df[performance_group_col] == g
        vals = metric[mask]
        n = int(mask.sum())
        table_rows.append(
            [
                g,
                range_strs[g],
                f"{n:,}",
                f"{100 * n / n_total:.1f}%",
                f"{float(vals.median()):.2f}",
                f"{float(vals.mean()):.2f}",
            ]
        )

    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(
        2,
        2,
        width_ratios=[3, 1],
        hspace=0.38,
        wspace=0.06,
        top=0.93,
        bottom=0.08,
        left=0.07,
        right=0.97,
    )
    ax_lin = fig.add_subplot(gs[0, 0])
    ax_tbl = fig.add_subplot(gs[0, 1])
    ax_log = fig.add_subplot(gs[1, :])

    # Linear histogram clipped at p99.5
    for g in _GROUP_ORDER:
        vals = metric[analysis_df[performance_group_col] == g]
        ax_lin.hist(
            vals,
            bins=80,
            range=(0, clip_val),
            alpha=0.85,
            label=g,
            color=_GROUP_COLORS[g],
            edgecolor="none",
        )
    y_top = ax_lin.get_ylim()[1]
    for bnd in [boundary_low, boundary_high]:
        ax_lin.axvline(min(bnd, clip_val), color="black", linestyle="--", linewidth=1.2)
        ax_lin.text(
            min(bnd, clip_val),
            y_top * 0.98,
            f"{bnd:.2f}",
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", pad=1),
        )
    ax_lin.set_xlim(0, clip_val)
    ax_lin.set_xlabel(f"{performance_metric_col} (m)")
    ax_lin.set_ylabel("Trajectory count")
    ax_lin.set_title(f"Distribution of {performance_metric_col} with 3 clusters")
    ax_lin.legend(title="Cluster", loc="center right")
    ax_lin.text(
        0.99,
        0.97,
        f"Linear view clipped at p99.5 = {clip_val:.2f} m\nFull max = {full_max:.2f} m",
        transform=ax_lin.transAxes,
        ha="right",
        va="top",
        fontsize=8.5,
        color="#555555",
    )

    # Cluster summary table
    ax_tbl.axis("off")
    ax_tbl.set_title(
        f"Cluster summary\nKMeans on log1p({performance_metric_col})",
        fontsize=9,
        loc="left",
    )
    col_labels = [
        "Cluster",
        f"{performance_metric_col} range",
        "n",
        "share",
        "median",
        "mean",
    ]
    tbl = ax_tbl.table(
        cellText=table_rows, colLabels=col_labels, cellLoc="center", loc="center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.auto_set_column_width(list(range(len(col_labels))))
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor("#e0e0e0")
            cell.set_text_props(weight="bold")
        else:
            cell.set_facecolor(
                to_rgba(_GROUP_COLORS[_GROUP_ORDER[row - 1]], alpha=0.35)
            )
        cell.set_edgecolor("#bbbbbb")

    # Log1p histogram (full tail view)
    log_metric = np.log1p(metric)
    log_min = float(log_metric.min())
    log_max = float(log_metric.max())
    for g in _GROUP_ORDER:
        vals_log = log_metric[analysis_df[performance_group_col] == g]
        ax_log.hist(
            vals_log,
            bins=80,
            range=(log_min, log_max),
            alpha=0.85,
            label=g,
            color=_GROUP_COLORS[g],
            edgecolor="none",
        )
    y_top_log = ax_log.get_ylim()[1]
    for bnd in [boundary_low, boundary_high]:
        ax_log.axvline(np.log1p(bnd), color="black", linestyle="--", linewidth=1.2)
        ax_log.text(
            np.log1p(bnd),
            y_top_log * 0.98,
            f"{bnd:.2f}",
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", pad=1),
        )
    raw_tick_candidates = [0.0001, 0.01, 0.1, 0.3, 1.0, 3.0, 6.0]
    raw_ticks = [t for t in raw_tick_candidates if np.log1p(t) <= log_max * 1.02]
    ax_log.set_xticks([np.log1p(t) for t in raw_ticks])
    ax_log.set_xticklabels([str(t) for t in raw_ticks])
    ax_log.set_xlabel(f"{performance_metric_col} (m), log1p-scaled axis")
    ax_log.set_ylabel("Count")
    ax_log.set_title("Full tail view on log1p scale")
    ax_log.legend(title="Cluster", loc="upper right")

    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_performance_group_distribution(
    analysis_df,
    performance_metric_col=raw_metric_col,
    performance_group_col=PERFORMANCE_GROUP_COL,
    group_summary_df=group_summary_df,
    plot_path=PERFORMANCE_GROUP_DISTRIBUTION_PATH,
)
print(
    f"Performance group distribution plot saved to: {PERFORMANCE_GROUP_DISTRIBUTION_PATH}"
)

## Evaluate Optional Reduced Representations
**Purpose:** When enabled, compute trustworthiness scores for dimensions `1, 2, ..., (#features - 1)` for each performance group and highlight the configured reduced dimension used for reduced-space clustering.<br>
**Inputs:** `analysis_df`, resolved `CLUSTER_SPEC_RESOLVED`, `PERFORMANCE_GROUP_COL`, and `effect_cols`.<br>
**Outputs:** `trustworthiness_df` and one trustworthiness curve per configured neighborhood view when optional UMAP reductions are enabled.<br>
**How to Verify:** If `CLUSTER_SPEC['evaluate_umap_latent_space']` is `False`, this section should emit an empty trustworthiness table and no trustworthiness plots.


In [ ]:
def plot_trustworthiness_curves(
    trustworthiness_df: pd.DataFrame,
    *,
    plot_titles: dict[str, str],
    plot_paths: dict[str, Path],
) -> None:
    for trustworthiness_view, plot_title in plot_titles.items():
        plot_df = trustworthiness_df.loc[
            trustworthiness_df["trustworthiness_view"] == trustworthiness_view
        ].copy()
        if plot_df.empty:
            raise ValueError(
                f"Missing trustworthiness rows for view={trustworthiness_view!r}."
            )

        fig, ax = plt.subplots(figsize=(10, 6.5))
        sns.lineplot(
            data=plot_df,
            x="n_components",
            y="trustworthiness",
            hue="performance_group",
            style="performance_group",
            markers=True,
            dashes=False,
            ax=ax,
        )
        selected_points_df = plot_df.loc[plot_df["selected_for_clustering"]].copy()
        if not selected_points_df.empty:
            ax.scatter(
                selected_points_df["n_components"],
                selected_points_df["trustworthiness"],
                s=90,
                facecolors="none",
                edgecolors="black",
                linewidths=1.2,
                zorder=5,
            )
        ax.set_title(plot_title)
        ax.set_xlabel("Reduced dimension")
        ax.set_ylabel("Trustworthiness")
        ax.set_xticks(sorted(plot_df["n_components"].unique().tolist()))
        plt.tight_layout()
        plt.savefig(plot_paths[trustworthiness_view], dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)


if CLUSTER_SPEC_RESOLVED["evaluate_umap_latent_space"]:
    trustworthiness_df = evaluate_umap_trustworthiness_by_group(
        analysis_df,
        cluster_spec=CLUSTER_SPEC_RESOLVED,
        performance_group_col=PERFORMANCE_GROUP_COL,
        effect_cols=effect_cols,
    )
    plot_trustworthiness_curves(
        trustworthiness_df,
        plot_titles=TRUSTWORTHINESS_PLOT_TITLES,
        plot_paths=UMAP_TRUSTWORTHINESS_PLOT_PATHS,
    )
    print(
        "If the highlighted dimensions do not match your intended clustering setup, "
        "update CLUSTER_SPEC['umap_selected_n_components'] and rerun from the previous section."
    )
else:
    trustworthiness_df = pd.DataFrame(columns=TRUSTWORTHINESS_COLUMNS)
    print(
        "Skipped optional UMAP trustworthiness evaluation because "
        "CLUSTER_SPEC['evaluate_umap_latent_space']=False. The 2D visualization UMAP "
        "is still computed in the clustering section."
    )

display(trustworthiness_df)


## Cluster Within Performance Groups
**Purpose:** Cluster feature-effect vectors separately inside each performance group and keep every candidate run available for downstream export and inspection.<br>
**Inputs:** `analysis_df`, resolved `CLUSTER_SPEC_RESOLVED`, `PERFORMANCE_GROUP_COL`, and `effect_cols`.<br>
**Outputs:** Candidate cluster assignments, scored clustering runs, and a best-run summary driven only by DBCV ranking metadata.<br>
**How to Verify:** Confirm that each performance group has scored cluster runs and that `selected_for_group` marks exactly one best-ranked candidate per group.


In [ ]:
clustering_results = run_step2_clustering(
    analysis_df,
    cluster_spec=CLUSTER_SPEC_RESOLVED,
    performance_group_col=PERFORMANCE_GROUP_COL,
    row_id_col="row_id",
    effect_cols=effect_cols,
)

clustered_df = clustering_results["clustered_df"]
cluster_scores_df = clustering_results["cluster_scores_df"]
best_cluster_runs_df = cluster_scores_df.loc[
    cluster_scores_df["selected_for_group"]
].copy()

print(f"Cluster score rows: {len(cluster_scores_df)}")
print(f"Best-ranked regime runs: {len(best_cluster_runs_df)}")

display(cluster_scores_df)
display(best_cluster_runs_df)

## Compare Clustering Outputs
**Purpose:** Show one panel per `(performance group, clustering algorithm)` combination for raw feature-effect space and, when enabled, reduced feature-effect spaces. Also save the same raw-space grid with noise labels excluded.<br>
**Inputs:** `cluster_scores_df`, `clustered_df`, resolved `CLUSTER_SPEC_RESOLVED`, and the shared visualization embedding stored in `clustered_df`.<br>
**Outputs:** Saved comparison grids, a raw-space no-noise comparison grid, and compact summary tables for the raw and reduced clustering runs.<br>
**How to Verify:** Confirm that each visible panel uses the expected performance group and algorithm, that the DBCV and cluster-count annotations line up with `cluster_scores_df`, and that the no-noise grid omits label `-1` while preserving the same group-by-algorithm layout.


In [ ]:
def plot_cluster_comparison_grid(
    cluster_scores_subset: pd.DataFrame,
    *,
    plot_path: Path,
    cluster_space_label: str,
    include_noise: bool = True,
) -> pd.DataFrame:
    groups = CLUSTER_SPEC_RESOLVED["groups"]
    algorithms = CLUSTER_SPEC_RESOLVED["algorithms"]
    comparison_df = cluster_scores_subset.copy()
    fig, axes = plt.subplots(
        len(groups),
        len(algorithms),
        figsize=(7.2 * len(algorithms), 4.8 * len(groups)),
        squeeze=False,
    )
    plotted_rows = []

    for row_idx, performance_group in enumerate(groups):
        for col_idx, algorithm in enumerate(algorithms):
            ax = axes[row_idx][col_idx]
            selected_rows = comparison_df.loc[
                (comparison_df["performance_group"] == performance_group)
                & (comparison_df["algorithm"] == algorithm)
            ]
            if selected_rows.empty:
                ax.set_visible(False)
                continue

            comparison_row = selected_rows.iloc[0]
            group_df = clustered_df.loc[
                clustered_df[PERFORMANCE_GROUP_COL] == performance_group
            ].copy()
            label_col = comparison_row["candidate_label_col"]
            cluster_ids = [
                int(cluster_id)
                for cluster_id in group_df[label_col]
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            ]
            ordered_cluster_ids = sorted(
                cluster_id for cluster_id in cluster_ids if cluster_id != -1
            )
            if include_noise and -1 in cluster_ids:
                ordered_cluster_ids.append(-1)

            noise_count = int((group_df[label_col].astype("Int64") == -1).sum())
            plotted_count = int(len(group_df) if include_noise else len(group_df) - noise_count)
            plotted_rows.append(
                {
                    "performance_group": performance_group,
                    "algorithm": algorithm,
                    "cluster_space": comparison_row["cluster_space"],
                    "candidate_label_col": label_col,
                    "plotted_rows": plotted_count,
                    "excluded_noise_rows": 0 if include_noise else noise_count,
                }
            )

            if not ordered_cluster_ids:
                ax.text(
                    0.5,
                    0.5,
                    "No non-noise\ncluster rows",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                )
                ax.set_axis_off()
                continue

            palette = sns.color_palette(
                "tab10",
                n_colors=max(
                    len(ordered_cluster_ids) - (1 if -1 in ordered_cluster_ids else 0),
                    1,
                ),
            )
            color_lookup = {
                cluster_id: palette[idx % len(palette)]
                for idx, cluster_id in enumerate(
                    cluster_id for cluster_id in ordered_cluster_ids if cluster_id != -1
                )
            }
            if -1 in ordered_cluster_ids:
                color_lookup[-1] = (0.65, 0.65, 0.65)

            for cluster_id in ordered_cluster_ids:
                cluster_rows = group_df.loc[
                    group_df[label_col].astype("Int64") == cluster_id
                ]
                ax.scatter(
                    cluster_rows["viz_umap_x"],
                    cluster_rows["viz_umap_y"],
                    s=28,
                    alpha=0.88,
                    c=[color_lookup[cluster_id]],
                    edgecolors="none",
                )

            dbcv_cluster_space = comparison_row["dbcv_cluster_space"]
            dbcv_cluster_label = (
                f"{dbcv_cluster_space:.3f}" if pd.notna(dbcv_cluster_space) else "NaN"
            )
            panel_title = (
                f"{algorithm.upper()}\n"
                f"DBCV={dbcv_cluster_label}, clusters={int(comparison_row['n_clusters'])}"
            )
            if not include_noise:
                panel_title = f"{panel_title}\nnoise excluded: {noise_count}"
            ax.set_title(panel_title)
            ax.set_xlabel("UMAP 1")
            if col_idx == 0:
                ax.set_ylabel(f"{performance_group}\nUMAP 2")
            else:
                ax.set_ylabel("")

    noise_label = "" if include_noise else " (noise excluded)"
    fig.suptitle(
        f"Cluster comparison in {cluster_space_label}{noise_label}",
        fontsize=16,
        y=1.01,
    )
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return pd.DataFrame(plotted_rows)


raw_comparison_df = (
    cluster_scores_df.loc[cluster_scores_df["cluster_space"] == "raw"]
    .copy()
    .sort_values(["performance_group", "algorithm"])
    .reset_index(drop=True)
)
plot_cluster_comparison_grid(
    raw_comparison_df,
    plot_path=RAW_ALGORITHM_GRID_PATH,
    cluster_space_label="raw feature-effect space",
)
raw_comparison_no_noise_df = plot_cluster_comparison_grid(
    raw_comparison_df,
    plot_path=RAW_ALGORITHM_GRID_NO_NOISE_PATH,
    cluster_space_label="raw feature-effect space",
    include_noise=False,
)

if CLUSTER_SPEC_RESOLVED["evaluate_umap_latent_space"]:
    umap_comparison_df = (
        cluster_scores_df.loc[cluster_scores_df["cluster_space"] == "umap"]
        .copy()
        .sort_values(["performance_group", "algorithm"])
        .reset_index(drop=True)
    )
    if umap_comparison_df.empty:
        raise ValueError(
            "Reduced-space clustering results are not available even though "
            "CLUSTER_SPEC['evaluate_umap_latent_space']=True."
        )
    plot_cluster_comparison_grid(
        umap_comparison_df,
        plot_path=UMAP_ALGORITHM_GRID_PATH,
        cluster_space_label="reduced feature-effect space",
    )
else:
    umap_comparison_df = pd.DataFrame(columns=cluster_scores_df.columns)
    print(
        "Reduced-space comparison grid skipped because CLUSTER_SPEC['evaluate_umap_latent_space']=False."
    )

print("Raw-space clustering runs:")
display(
    raw_comparison_df[
        [
            "performance_group",
            "algorithm",
            "cluster_space",
            "dbcv_raw_effect_space",
            "n_clusters",
            "noise_fraction",
        ]
    ]
)
print("Reduced-space clustering runs:")
display(
    umap_comparison_df[
        [
            "performance_group",
            "algorithm",
            "cluster_space",
            "dbcv_raw_effect_space",
            "n_clusters",
            "noise_fraction",
        ]
    ]
)
print("Raw-space no-noise UMAP rows:")
display(raw_comparison_no_noise_df)


## Assemble Candidate-Wide Cluster Exports
**Purpose:** Build lean downstream exports for every candidate cluster subset without duplicating the authoritative assignment table, and copy the run-level global feature-effect ranking into the cluster-spec export tree.<br>
**Inputs:** `clustered_df`, `cluster_scores_df`, resolved export paths, `raw_metric_col`, `effect_cols`, and `feature_effect_ranking_df`.<br>
**Outputs:** Candidate-wide `cluster_feature_effect_profiles_df`, `cluster_catalog_df`, one slim member file per exported cluster subset, and `feature_effect_global_ranking.csv` for downstream inspection ordering.<br>
**How to Verify:** Confirm that the catalog includes every `(performance group, algorithm, cluster space, cluster_id)` subset, including noise when present, and that the copied ranking table preserves the run-level global order.


In [ ]:
cluster_export_outputs = write_cluster_exports(
    clustered_df,
    cluster_scores_df,
    export_layout=EXPORT_LAYOUT,
    performance_metric_col=raw_metric_col,
    performance_group_col=PERFORMANCE_GROUP_COL,
    effect_cols=effect_cols,
)

cluster_feature_effect_profiles_df = cluster_export_outputs[
    "cluster_feature_effect_profiles_df"
]
cluster_catalog_df = cluster_export_outputs["cluster_catalog_df"]
feature_effect_ranking_df.to_csv(FEATURE_EFFECT_GLOBAL_RANKING_PATH, index=False)

if (
    cluster_export_outputs["cluster_feature_effect_profiles_path"]
    != CLUSTER_FEATURE_EFFECT_PROFILES_PATH
):
    raise ValueError(
        "Cluster feature-effect profile export path mismatch between notebook and export helper."
    )
if cluster_export_outputs["cluster_catalog_path"] != CLUSTER_CATALOG_PATH:
    raise ValueError(
        "Cluster catalog export path mismatch between notebook and export helper."
    )

print(f"Cluster profile rows: {len(cluster_feature_effect_profiles_df)}")
print(f"Cluster catalog rows: {len(cluster_catalog_df)}")
print(f"Copied global ranking rows: {len(feature_effect_ranking_df)}")

display(cluster_catalog_df.head())
display(cluster_feature_effect_profiles_df.head())
display(feature_effect_ranking_df.head())

## Export Regime Artifacts
**Purpose:** Persist the assembled regime table, clustering outputs, candidate-wide export artifacts, raw-space no-noise UMAP plot, and the copied global feature-effect ranking for downstream interpretation.<br>
**Inputs:** The completed notebook dataframes, export-helper outputs, and the plot paths accumulated in earlier sections.<br>
**Outputs:** CSV exports, saved plot files, and a manifest table that records only the main clustering artifacts required by downstream notebooks.<br>
**How to Verify:** Check that every printed output path exists under the run-specific results directory and that the manifest records the neutral feature-effect artifact names.


In [ ]:
analysis_df.to_csv(REGIME_ANALYSIS_PATH, index=False)
group_summary_df.to_csv(PERFORMANCE_GROUP_SUMMARY_PATH, index=False)
trustworthiness_df.to_csv(UMAP_TRUSTWORTHINESS_PATH, index=False)
cluster_scores_df.to_csv(CLUSTER_SCORES_PATH, index=False)
# `cluster_assignments.csv` stays authoritative because it keeps the full joined table, feature effects,
# visualization embedding, and every candidate label column in one place.
clustered_df.to_csv(CLUSTER_ASSIGNMENTS_PATH, index=False)

current_artifact_records = [
    {
        "artifact_kind": "table",
        "artifact_type": "regime_analysis",
        "relative_path": str(
            REGIME_ANALYSIS_PATH.relative_to(EXPORT_LAYOUT["cluster_spec_root"])
        ),
        "absolute_path": str(REGIME_ANALYSIS_PATH.resolve()),
    },
    {
        "artifact_kind": "table",
        "artifact_type": "performance_group_summary",
        "relative_path": str(
            PERFORMANCE_GROUP_SUMMARY_PATH.relative_to(
                EXPORT_LAYOUT["cluster_spec_root"]
            )
        ),
        "absolute_path": str(PERFORMANCE_GROUP_SUMMARY_PATH.resolve()),
    },
    {
        "artifact_kind": "table",
        "artifact_type": "umap_trustworthiness",
        "relative_path": str(
            UMAP_TRUSTWORTHINESS_PATH.relative_to(EXPORT_LAYOUT["cluster_spec_root"])
        ),
        "absolute_path": str(UMAP_TRUSTWORTHINESS_PATH.resolve()),
    },
    {
        "artifact_kind": "table",
        "artifact_type": "cluster_scores",
        "relative_path": str(
            CLUSTER_SCORES_PATH.relative_to(EXPORT_LAYOUT["cluster_spec_root"])
        ),
        "absolute_path": str(CLUSTER_SCORES_PATH.resolve()),
    },
    {
        "artifact_kind": "table",
        "artifact_type": "cluster_assignments",
        "relative_path": str(
            CLUSTER_ASSIGNMENTS_PATH.relative_to(EXPORT_LAYOUT["cluster_spec_root"])
        ),
        "absolute_path": str(CLUSTER_ASSIGNMENTS_PATH.resolve()),
    },
    {
        "artifact_kind": "table",
        "artifact_type": "feature_effect_global_ranking",
        "relative_path": str(
            FEATURE_EFFECT_GLOBAL_RANKING_PATH.relative_to(
                EXPORT_LAYOUT["cluster_spec_root"]
            )
        ),
        "absolute_path": str(FEATURE_EFFECT_GLOBAL_RANKING_PATH.resolve()),
    },
    {
        "artifact_kind": "plot",
        "artifact_type": "raw_algorithm_comparison_grid",
        "relative_path": str(
            RAW_ALGORITHM_GRID_PATH.relative_to(EXPORT_LAYOUT["cluster_spec_root"])
        ),
        "absolute_path": str(RAW_ALGORITHM_GRID_PATH.resolve()),
    },
    {
        "artifact_kind": "plot",
        "artifact_type": "raw_algorithm_comparison_grid_no_noise",
        "relative_path": str(
            RAW_ALGORITHM_GRID_NO_NOISE_PATH.relative_to(
                EXPORT_LAYOUT["cluster_spec_root"]
            )
        ),
        "absolute_path": str(RAW_ALGORITHM_GRID_NO_NOISE_PATH.resolve()),
    },
]
if CLUSTER_SPEC_RESOLVED["evaluate_umap_latent_space"]:
    current_artifact_records.append(
        {
            "artifact_kind": "plot",
            "artifact_type": "umap_algorithm_comparison_grid",
            "relative_path": str(
                UMAP_ALGORITHM_GRID_PATH.relative_to(EXPORT_LAYOUT["cluster_spec_root"])
            ),
            "absolute_path": str(UMAP_ALGORITHM_GRID_PATH.resolve()),
        }
    )
for trustworthiness_view, plot_path in UMAP_TRUSTWORTHINESS_PLOT_PATHS.items():
    current_artifact_records.append(
        {
            "artifact_kind": "plot",
            "artifact_type": "umap_trustworthiness_curve",
            "trustworthiness_view": trustworthiness_view,
            "relative_path": str(
                plot_path.relative_to(EXPORT_LAYOUT["cluster_spec_root"])
            ),
            "absolute_path": str(plot_path.resolve()),
        }
    )
current_artifact_records.extend(cluster_export_outputs["artifact_records"])

manifest_data = load_or_initialize_feature_effect_regime_manifest(
    EXPORT_MANIFEST_PATH,
    run_context={
        "model_id": MODEL_ID,
        "run_name": RUN_NAME,
        "target_col": target_col,
        "target_mode": target_mode,
        "eval_csv_name": EVAL_CSV_NAME,
        "run_manifest_path": str(run_ctx.manifest_path),
        "prepared_data_path": str(PREPARED_DATA_PATH),
        "feature_effects_path": str(FEATURE_EFFECTS_PATH),
        "feature_effect_importance_path": str(FEATURE_EFFECT_IMPORTANCE_PATH),
        "joined_metrics_path": str(JOINED_METRICS_PATH),
    },
    data_context={
        "target_col": target_col,
        "eval_csv_name": EVAL_CSV_NAME,
        "lower_is_better": LOWER_IS_BETTER,
        "performance_group_col": PERFORMANCE_GROUP_COL,
        "data_context_slug": EXPORT_CONTEXT["data_context_slug"],
    },
    cluster_spec={
        "raw": CLUSTER_SPEC,
        "resolved": CLUSTER_SPEC_RESOLVED,
        "readable_slug": EXPORT_LAYOUT["cluster_spec_readable_slug"],
        "hash": EXPORT_LAYOUT["cluster_spec_hash"],
        "dirname": EXPORT_LAYOUT["cluster_spec_dirname"],
    },
)
manifest_data = merge_feature_effect_regime_artifact_records(
    manifest_data,
    artifact_records=current_artifact_records,
)
EXPORT_MANIFEST_PATH.write_text(json.dumps(manifest_data, indent=2))
artifact_manifest_df = pd.DataFrame(manifest_data["artifacts"])

print("Saved artifacts:")
print(f"- Run manifest:                 {run_ctx.manifest_path}")
print(f"- Prepared data export:         {PREPARED_DATA_PATH}")
print(f"- Feature-effect export:        {FEATURE_EFFECTS_PATH}")
print(f"- Feature-effect importance:    {FEATURE_EFFECT_IMPORTANCE_PATH}")
print(f"- Joined metrics export:        {JOINED_METRICS_PATH}")
print(f'- Cluster-spec export root:     {EXPORT_LAYOUT["cluster_spec_root"]}')
print(f"- Export manifest:              {EXPORT_MANIFEST_PATH}")
print(f"- Regime analysis table:        {REGIME_ANALYSIS_PATH}")
print(f"- Performance group summary:    {PERFORMANCE_GROUP_SUMMARY_PATH}")
print(f"- UMAP trustworthiness:         {UMAP_TRUSTWORTHINESS_PATH}")
for trustworthiness_view, plot_path in UMAP_TRUSTWORTHINESS_PLOT_PATHS.items():
    print(f"- Trustworthiness curve ({trustworthiness_view}): {plot_path}")
print(f"- Cluster scores:               {CLUSTER_SCORES_PATH}")
print(f"- Cluster assignments:          {CLUSTER_ASSIGNMENTS_PATH}")
print(f"- Cluster feature-effect profiles: {CLUSTER_FEATURE_EFFECT_PROFILES_PATH}")
print(f"- Cluster catalog:              {CLUSTER_CATALOG_PATH}")
print(f"- Copied global ranking:        {FEATURE_EFFECT_GLOBAL_RANKING_PATH}")
print(f"- Raw comparison grid:          {RAW_ALGORITHM_GRID_PATH}")
print(f"- Raw no-noise comparison grid: {RAW_ALGORITHM_GRID_NO_NOISE_PATH}")
if CLUSTER_SPEC_RESOLVED["evaluate_umap_latent_space"]:
    print(f"- Reduced comparison grid:      {UMAP_ALGORITHM_GRID_PATH}")
else:
    print(
        "- Reduced comparison grid:      skipped (CLUSTER_SPEC['evaluate_umap_latent_space']=False)"
    )
print(f"- Tables directory:             {TABLES_DIR}")
print(f"- Plot directory:               {PLOTS_DIR}")

display(artifact_manifest_df)
